<a href="https://colab.research.google.com/github/Shahana023/cse-resources/blob/main/autonomous_vit/YOLOv5_INRIA_patch_attack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg"/></a>

# Adversarial Patch Attack on YOLOv5 Person Detection (INRIA)

A small **wearable-patch** attack: we train one patch on the **INRIA Person** dataset so that, when it is placed on a person, **YOLOv5 stops detecting them**. Real dataset, real detector, measured with detection metrics.

The method follows two recent papers on physically realizable, transferable patches for driving / pedestrian detection — **AdvAD** (arXiv:2604.23105, 2026) and **TriPatch** (arXiv:2604.22552, 2026). This is a compact demonstration of that attack, not a new method.

**Runtime:** a few minutes on a **GPU** (Colab: Runtime -> Change runtime type -> GPU; Kaggle: Settings -> Accelerator -> GPU).

## Step 0 — Setup

In [ ]:
# extra libraries: ultralytics (used by the YOLOv5 loader) and datasets (pulls INRIA)
# torch/torchvision are already present on Colab and Kaggle
!pip install -q ultralytics datasets

import warnings; warnings.filterwarnings("ignore")
import torch, torch.nn.functional as F, numpy as np, random
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from torchvision.ops import nms
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print("running on:", device)

## Step 1 — Load a pretrained YOLOv5

In [ ]:
# YOLOv5-small, pretrained on COCO (so it already knows the "person" class)
model = torch.hub.load("ultralytics/yolov5", "yolov5s", pretrained=True, trust_repo=True)
det = model.model.float().to(device).eval()      # the raw network, so we can back-propagate to the image
for p in det.parameters():
    p.requires_grad_(False)                       # we optimise the patch, never the model weights

PERSON = 0                                        # "person" is class 0 in COCO

def raw(z):
    # predictions BEFORE non-max-suppression: shape [batch, num_boxes, 85]
    # 85 = 4 box coords + 1 objectness + 80 class scores
    o = det(z)
    return o[0] if isinstance(o, (list, tuple)) else o

def person_conf(z):
    # how confident each box is that it is a person = objectness * person-class score
    pr = raw(z)
    return pr[..., 4] * pr[..., 5 + PERSON]

def person_boxes(x, th=0.25):
    # the person boxes YOLOv5 actually keeps after NMS (used for display + scoring)
    with torch.no_grad():
        pr = raw(x)[0]
    pc = pr[:, 4] * pr[:, 5 + PERSON]
    keep = pc > th
    p, sc = pr[keep], pc[keep]
    if len(p) == 0:
        return []
    xy, wh = p[:, :2], p[:, 2:4]
    boxes = torch.cat([xy - wh / 2, xy + wh / 2], 1)   # convert xywh -> xyxy
    return boxes[nms(boxes, sc, 0.45)].tolist()

print("YOLOv5 ready")

## Step 2 — Load the INRIA Person dataset

In [ ]:
def to_tensor(im):
    # every image -> 640x640 tensor, pixels in [0,1] (the input YOLOv5 expects)
    im = im.convert("RGB").resize((640, 640))
    return torch.from_numpy(np.array(im).astype("float32") / 255).permute(2, 0, 1).unsqueeze(0).to(device)

# INRIA from a public HuggingFace mirror (no login needed).
# If it is ever down, the same data is on Kaggle: jcoral02/inriaperson
data = load_dataset("marcelarosalesj/inria-person", split="train", streaming=True)

imgs, boxes = [], []
N_WANT = 60                                       # how many usable person images to gather
for ex in data:
    if ex["label"] != 1:                          # label 1 = "pedestrians"; skip empty-street photos
        continue
    x = to_tensor(ex["image"])
    b = person_boxes(x)
    if len(b) >= 1:                               # keep only images where YOLOv5 really sees a person
        imgs.append(x); boxes.append(b)
    if len(imgs) >= N_WANT:
        break

split = int(0.7 * len(imgs))
train_idx, eval_idx = list(range(split)), list(range(split, len(imgs)))
print(f"collected {len(imgs)} INRIA person images  ->  {len(train_idx)} train / {len(eval_idx)} eval")

## Step 3 — Baseline detection (before any attack)

In [ ]:
def show(ax, x, bxs, title):
    ax.imshow(x[0].permute(1, 2, 0).detach().cpu().numpy()); ax.set_title(title); ax.axis("off")
    for (x1, y1, x2, y2) in bxs:
        ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, lw=2, edgecolor="lime", facecolor="none"))

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, i in zip(axs, eval_idx[:3]):
    bxs = person_boxes(imgs[i])
    show(ax, imgs[i], bxs, f"clean: {len(bxs)} person(s)")
plt.tight_layout(); plt.show()

## Step 4 — The patch, and how it is placed

The patch is pasted onto each detected person. During training we jitter its size, brightness and position a little every step — this is **Expectation over Transformation (EOT)**, and it is what makes a patch survive being printed and worn in the real world.

In [ ]:
def apply_patch(img, bxs, patch, train=True):
    out = img.clone(); _, _, H, W = img.shape
    for (x1, y1, x2, y2) in bxs:
        bh = y2 - y1
        s = int(0.45 * bh)                        # patch height ~ 45% of the person's height
        if s < 8:
            continue
        if train:                                 # EOT: random scale / brightness / position
            s = int(s * (0.9 + 0.2 * torch.rand(1).item()))
            bright = 0.8 + 0.4 * torch.rand(1).item()
            jx = int((torch.rand(1).item() - 0.5) * 0.1 * bh)
            jy = int((torch.rand(1).item() - 0.5) * 0.1 * bh)
        else:
            bright, jx, jy = 1.0, 0, 0
        p = F.interpolate(patch.unsqueeze(0), size=(s, s), mode="bilinear", align_corners=False)[0]
        p = (p * bright).clamp(0, 1)
        cx = int((x1 + x2) / 2) + jx
        cy = int(y1 + 0.35 * bh) + jy             # upper torso, where a worn patch would sit
        y0, x0 = max(0, cy - s // 2), max(0, cx - s // 2)
        ye, xe = min(H, y0 + s), min(W, x0 + s)
        ph, pw = ye - y0, xe - x0
        if ph > 0 and pw > 0:
            out[:, :, y0:ye, x0:xe] = p[:, :ph, :pw]
    return out

def tv(p):
    # total-variation: keeps the patch smooth, which also makes it more printable (used in both papers)
    return (p[:, 1:, :] - p[:, :-1, :]).abs().mean() + (p[:, :, 1:] - p[:, :, :-1]).abs().mean()

## Step 5 — Train one universal patch

We train a single patch to push down YOLOv5's person-confidence across the training people. The loss is the mean of the strongest person scores (plus the smoothness term).

In [ ]:
patch = torch.rand(3, 200, 200, device=device, requires_grad=True)
opt = torch.optim.Adam([patch], lr=0.03)

STEPS, BATCH = 300, 6            # GPU: a few minutes. Use fewer steps for a quick look, more for a stronger patch.
for step in range(STEPS):
    idx = random.sample(train_idx, min(BATCH, len(train_idx)))
    loss = 0
    for i in idx:
        pc = person_conf(apply_patch(imgs[i], boxes[i], patch, train=True))
        loss = loss + pc.topk(min(10, pc.shape[1]), dim=1).values.mean()   # suppress the top person scores
    loss = loss / len(idx) + 0.05 * tv(patch)
    opt.zero_grad(); loss.backward(); opt.step()
    patch.data.clamp_(0, 1)                        # keep the patch a valid image
    if step % 50 == 0:
        print(f"step {step:4d}   loss {loss.item():.4f}")

patch_final = patch.detach()
plt.figure(figsize=(3, 3)); plt.imshow(patch_final.permute(1, 2, 0).cpu().numpy())
plt.title("the trained patch"); plt.axis("off"); plt.show()

## Step 6 — See the attack (before / after)

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(15, 10))
for col, i in enumerate(eval_idx[:3]):
    show(axs[0, col], imgs[i], person_boxes(imgs[i]), f"before: {len(person_boxes(imgs[i]))} person(s)")
    adv = apply_patch(imgs[i], boxes[i], patch_final, train=False)
    show(axs[1, col], adv, person_boxes(adv), f"with patch: {len(person_boxes(adv))} person(s)")
plt.tight_layout(); plt.show()

## Step 7 — Metrics beyond accuracy

Scoring the attack the way a detection paper would: how many people vanish (Attack Success Rate), how many survive (recall), and how far the confidence drops.

In [ ]:
n_clean = n_att = c_clean = c_att = seen = 0
for i in eval_idx:
    b0 = person_boxes(imgs[i])
    if not b0:
        continue
    adv = apply_patch(imgs[i], boxes[i], patch_final, train=False)
    b1 = person_boxes(adv)
    n_clean += len(b0); n_att += len(b1); seen += 1
    c_clean += person_conf(imgs[i]).max().item()
    c_att   += person_conf(adv).max().item()

asr = (n_clean - n_att) / max(n_clean, 1)
print(f"eval images                 : {seen}")
print(f"persons detected clean->atk : {n_clean} -> {n_att}")
print(f"Attack Success Rate (ASR)   : {asr*100:.1f}%")
print(f"recall retained             : {n_att / max(n_clean, 1) * 100:.1f}%")
print(f"mean max person-confidence  : {c_clean / max(seen, 1):.3f} -> {c_att / max(seen, 1):.3f}")

## What this shows — and its limits

- **Result.** One patch, trained on INRIA people, makes YOLOv5 miss most of them on held-out images — measured by Attack Success Rate and the drop in person-confidence, not just accuracy.
- **Method.** Objectness x person-score suppression, with Expectation-over-Transformation (random scale / brightness / position) and a total-variation smoothness term — the recipe described by **AdvAD** and **TriPatch**.
- **Physical world.** The EOT jitter is what lets such a patch survive being *printed and worn*; the two papers add rotation / perspective and optimise across several detectors so one patch transfers.
- **Limits (stated honestly).** This is a demonstration with a short digital optimization against a single detector — not a novel method. A stronger patch needs more images and iterations; real-world use needs printing, non-printability constraints, and physical testing.

**References.** AdvAD — *Transferable Physical-World Adversarial Patches Against Object Detection in Autonomous Driving* (arXiv:2604.23105, 2026). TriPatch — *Transferable Physical-World Adversarial Patches Against Pedestrian Detection Models* (arXiv:2604.22552, 2026). Foundational patch attack — Thys, Van Ranst & Goedemé, *Fooling automated surveillance cameras* (arXiv:1904.08653, CVPRW 2019).